---

## pdf 파일 전처리

**파일 종류**
- AI 사업문서
- AI 관련 훈령 문서

**현재 상태**
- 원본 pdf 거나 처리의 용이를 위해 pdf로 변환된(hwp,hwpx 파일)

**처리 방법**
- 텍스트 추출
- 표는 마크다운 형식 표로 변환하여 텍스트에 추가
- 메타데이터 추출하여 첨부

In [3]:
import pdfplumber
import os
import json

class PDFExtraction:
    def __init__(self, pdf_path):
        self.pdf_path = pdf_path
    
    def extract_text(self, source=None, category=None):
        result = []
        with pdfplumber.open(self.pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages, start=1):
                text = ""
                if page.extract_text():
                    text += page.extract_text() + "\n"

                tables = page.extract_tables()
                if tables:
                    for table in tables:
                        md_table = self.extract_tables(table)
                        text += f"\n```markdown\n{md_table}\n```"

                result.append({
                    'source': source,
                    'category': category,
                    'page': page_num,
                    'text': text
                })

        return result

    def extract_tables(self, table):
        if not table or not table[0]:
            return ""
        max_cols = max(len(row) for row in table if row)
        normalize = lambda row: row + [""] * (max_cols - len(row))
        table = [normalize(row) for row in table]
        header = "| " + " | ".join([str(cell) if cell is not None else "" for cell in table[0]]) + " |"
        separator = "| " + " | ".join(['---'] * len(table[0])) + " |"
        rows = [
            "| " + " | ".join([str(cell) if cell is not None else "" for cell in row]) + " |"
            for row in table[1:]
        ]
        return "\n".join([header, separator] + rows)


def process_folder(input_root, output_root):
    for root, dirs, files in os.walk(input_root):
        for filename in files:
            if not filename.lower().endswith(".pdf"):
                continue

            input_path = os.path.join(root, filename)

            # 상대 경로 추출
            relative_path = os.path.relpath(root, input_root)
            output_dir = os.path.join(output_root, relative_path)
            os.makedirs(output_dir, exist_ok=True)

            # 출력 파일 경로
            output_filename = os.path.splitext(filename)[0] + ".json"
            output_path = os.path.join(output_dir, output_filename)

            #메타데이터 추출
            category = os.path.basename(root)
            source = filename

            # 처리
            extractor = PDFExtraction(input_path)
            result = extractor.extract_text(source=source, category=category)

            with open(output_path, 'w', encoding='utf-8') as f:
                json.dump(result, f, ensure_ascii=False, indent=2)



In [18]:
# 사용 예시
# PDF가 있는 폴더: ./data/pdfs
# 결과 저장 폴더: ./data/parsed

process_folder('C:/SKN AI CAMP TEAM/SKN09-FINAL-1Team/model_hosting/data/origin_data', 'C:/SKN AI CAMP TEAM/SKN09-FINAL-1Team/model_hosting/data/preprocess')

CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, def

In [4]:
# 사용 예시
# PDF가 있는 폴더: ./data/pdfs
# 결과 저장 폴더: ./data/parsed

process_folder('C:/SKN AI CAMP TEAM/SKN09-FINAL-1Team/model_hosting/data/origin_data/훈령', 'C:/SKN AI CAMP TEAM/SKN09-FINAL-1Team/model_hosting/data/preprocess/훈령')

CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, def

---

## 법령 JSON 데이터 전처리

**파일 종류**
- 과기부 소관 AI산업 관련 법령 47종

**현재 상태**
- 기본 처리 된 상태 (xml_to_json, 조문별 구분)
- 메타데이터 분리되어 첫번째 오브젝트로 들어가 있음

**처리 방법**
- 메타데이터를 각 오브젝트에 동일하게 추가(단 pdf파일과는 메타데이터 양식 다름)
- 조문 전체가 삭제된 항목은 탐지하여 제거
- 일부 항/호/목 만 삭제 된 경우 전체 항목 유지

In [1]:
import json
import re
from pathlib import Path

# 삭제된 조문 체크
def is_fully_deleted_article(key: str, value: str) -> bool:
    pattern = r"^제\d+조(?:의\d+)?\s*삭제\s*<\d{4}\.\d{1,2}\.\d{1,2}>[\s.]*$"
    return bool(re.fullmatch(pattern, value.strip()))


def process_json_file(file_path: Path, output_dir: Path):
    with file_path.open(encoding="utf-8") as f:
        data = json.load(f)

    # 첫 객체에서 메타데이터 추출
    meta = next((item for item in data if isinstance(item, dict) and "법령명" in item), {})
    metadata = {
        "법령명": meta.get("법령명"),
        "법종구분": meta.get("법종구분"),
        "공포일자": meta.get("공포일자"),
        "시행일자": meta.get("시행일자"),
        "소관부처": meta.get("소관부처"),
    }

    result = []
    # 첫 번째 오브젝트를 제외한 나머지만 순회
    for item in data[1:]:
        if not isinstance(item, dict):
            continue
        for key, value in item.items():
            if key.startswith("제") and isinstance(value, str):
                if is_fully_deleted_article(key, value):
                    continue
                result.append({
                    "metadata": metadata,
                    "text": value
                })
    

    # 결과 저장
    output_path = output_dir / f"{file_path.stem}_qdrant.json"
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

def process_all_json_in_folder(input_folder: str, output_folder: str):
    input_dir = Path(input_folder)
    output_dir = Path(output_folder)
    output_dir.mkdir(parents=True, exist_ok=True)

    for json_file in input_dir.glob("*.json"):
        process_json_file(json_file, output_dir)
        print(f"✅ Processed: {json_file.name}")


In [2]:
# 사용 예시
process_all_json_in_folder("C:/SKN AI CAMP TEAM/SKN09-FINAL-1Team/model_hosting/data/origin_data/법령", "C:/SKN AI CAMP TEAM/SKN09-FINAL-1Team/model_hosting/data/preprocess/법령")

✅ Processed: 4차산업혁명위원회의설치및운영에관한규정_228675.json
✅ Processed: 가상융합산업진흥법_260801.json
✅ Processed: 가상융합산업진흥법시행규칙_265103.json
✅ Processed: 가상융합산업진흥법시행령_265063.json
✅ Processed: 과학기술정보통신부소관비상대비에관한법률시행규칙_255981.json
✅ Processed: 과학기술정보통신부와그소속기관직제시행규칙_269413.json
✅ Processed: 과학기술정보통신부장관의소속청장지휘에관한규칙_266063.json
✅ Processed: 국가인공지능위원회의설치및운영에관한규정_264599.json
✅ Processed: 국가지식정보연계및활용촉진에관한법률_232613.json
✅ Processed: 국가지식정보연계및활용촉진에관한법률시행령_237337.json
✅ Processed: 국가초고성능컴퓨터활용및육성에관한법률_258989.json
✅ Processed: 국가초고성능컴퓨터활용및육성에관한법률시행규칙_244011.json
✅ Processed: 국가초고성능컴퓨터활용및육성에관한법률시행령_243665.json
✅ Processed: 데이터산업진흥및이용촉진에관한기본법_236051.json
✅ Processed: 데이터산업진흥및이용촉진에관한기본법시행규칙_247551.json
✅ Processed: 데이터산업진흥및이용촉진에관한기본법시행령_265725.json
✅ Processed: 방송법시행규칙_270879.json
✅ Processed: 방송통신설비의기술기준에관한규정_263865.json
✅ Processed: 소프트웨어진흥법_265845.json
✅ Processed: 소프트웨어진흥법시행규칙_271031.json
✅ Processed: 소프트웨어진흥법시행령_270707.json
✅ Processed: 전기통신기본법_206003.json
✅ Processed: 전기통신기본법시행령_109707.json
✅ Processed: 전자서명법_236201